In [1]:
import torch
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
import pickle as pkl
import random
from itertools import product

# Data loading

In [37]:
with open("./data/ml_ready_data.pickle", "br") as f:
    data = pkl.load(f)

X_ab, X_hero, Y, ab_id2idx, ab_idx2id, hero_id2idx, hero_idx2id, match_ids = data

class ad_data(torch.utils.data.Dataset):
    def __init__(self, X_ab, X_hero, Y):
        super(ad_data, self).__init__()
        self.X_ab = X_ab
        self.X_hero = X_hero
        self.Y = Y

    def __len__(self):
        return self.Y.shape[0]

    def __getitem__(self, idx):
        return X_ab[idx, :, :], X_hero[idx, :], Y[idx]


n_ab = len(ab_idx2id)
n_hero = len(hero_idx2id)
n_sample = X_ab.shape[0]

In [3]:
n_train = 10000
n_test = 10000

np.random.seed(1111)

idx = np.arange(n_sample)
np.random.shuffle(idx)

idx_train = idx[0 : n_train]
idx_test = idx[n_train : n_train+n_test]

X_ab = X_ab.astype(np.int64)
X_hero = X_hero.astype(np.int64)
Y = Y.astype(np.float64)

train_data = ad_data(X_ab[idx_train, :, :], X_hero[idx_train, :], Y[idx_train])
test_data = ad_data(X_ab[idx_test, :, :], X_hero[idx_test, :], Y[idx_test])

# Define models

### attention-type player encoder: $x_{player} = \sum_{l, ab} \text{softmax}_{ab}\left(v_{hero}^T A_l v_{ab}\right)B_l v_{ab}$

In [4]:
class player_attention(torch.nn.Module):
    def __init__(self, dim_hero, dim_ab, dim_out, n_head):
        super(player_attention, self).__init__()
        self.dim_hero = dim_hero
        self.dim_ab = dim_ab
        self.dim_out = dim_out
        self.n_head = n_head
        self.As = torch.nn.ModuleList([torch.nn.Linear(dim_ab, dim_hero, bias=False) for _ in range(n_head)])
        self.Bs = torch.nn.ModuleList([torch.nn.Linear(dim_ab, dim_out, bias=False) for _ in range(n_head)])
    
    def forward(self, x_ab, x_hero):
        # x_ab: (n_batch, n_item, dim_ab) sized tensor
        # x_hero:  (n_batch, dim_hero)
        n_batch, n_item, _ = x_ab.shape
        tmp = []
        for i in range(self.n_head):
            z = self.As[i](x_ab) # z: (n_batch, n_item, h_dim)
            z = (z * x_hero.view(n_batch, 1, self.dim_hero)).sum(dim=2) / np.sqrt(self.dim_hero)
            z = torch.softmax(z, dim=1) # z: (n_batch, n_item)
            z2 = self.Bs[i](x_ab) # z2: (n_batch, n_item, dim_out)
            z = (z.view(n_batch, n_item, 1) * z2).sum(dim=1)
            tmp.append(z)
        z_player = sum(tmp) / np.sqrt(self.n_head)
        return z_player


class team_sum(torch.nn.Module):
    def __init__(self):
        super(team_sum, self).__init__()
    
    def forward(self, x_player):
        # x_player: (n_batch, n_player, player_dim)
        n_batch, n_player, player_dim = x_player.shape
        z = x_player.sum(dim=1) / np.sqrt(n_player)
        return z

class simple_decoder(torch.nn.Module):
    def __init__(self):
        super(simple_decoder, self).__init__()

    def forward(self, x_team1, x_team2):
        # X_team_1/2: (n_batch, dim_team)
        n_batch, dim_team = x_team1.shape
        z = (x_team1 - x_team2).sum(dim=1)/np.sqrt(dim_team)
        return z

# Run

In [5]:
def seed_all(sd):
    torch.manual_seed(sd)
    np.random.seed(sd)
    random.seed(sd)

class my_model(torch.nn.Module):
    def __init__(self, n_hero, n_ab, n_head, dim_hero, dim_ab, dim_player, dim_team):
        super(my_model, self).__init__()
        self.n_hero = n_hero
        self.n_head = n_head
        self.dim_hero = dim_hero
        self.dim_ab = dim_ab
        self.dim_player = dim_player
        self.dim_team = dim_team

        self.ab_embd = torch.nn.Embedding(n_ab, dim_ab)
        self.hero_embd = torch.nn.Embedding(n_hero, dim_hero)

        self.player_encoder = player_attention(dim_hero, dim_ab, dim_player, n_head)
        self.team_encoder = team_sum()
        self.decoder = simple_decoder()
    
    def forward(self, x_ab, x_hero):
        n_batch = x_ab.shape[0]
        z_ab = self.ab_embd(x_ab) # (n_batch, 10, 4, dim_ab)
        z_hero = self.hero_embd(x_hero) # (n_batch, 10, dim_hero)
        z_player = self.player_encoder(z_ab.view(n_batch*10, 4, -1), z_hero.view(n_batch*10, -1)).view(n_batch, 10, self.dim_player)
        z_rad = self.team_encoder(z_player[:, 0:5, :])
        z_dire = self.team_encoder(z_player[:, 5:10, :])
        z_game = self.decoder(z_rad, z_dire)

        return z_game

In [28]:
dim_hero = 8
dim_ab = 8
dim_player = 32
n_head = 4
dim_team = None
n_batch = 100

lr = 1
n_ep = 1

model = my_model(n_hero, n_ab, n_head, dim_hero, dim_ab, dim_player, dim_team)


if torch.cuda.is_available():
    device = torch.device("cuda:0")
else:
    device = torch.device("cpu")

model.to(device)
# train_data.to(device)
# test_data.to(device)
train_loader = torch.utils.data.DataLoader(train_data, batch_size=n_batch, shuffle=True)
test_loader = torch.utils.data.DataLoader(test_data, batch_size=n_test, shuffle=True)


optimizer = torch.optim.SGD(model.parameters(), lr=lr)
# scheduler = torch.optim.lr_scheduler.StepLR(optimizer, 1.0, gamma=0.95)

In [7]:
def train(model, device, train_loader, optimizer, epoch):
    model.train()
    for batch_idx, (x_ab, x_hero, y) in enumerate(train_loader):

        x_ab.to(device)
        x_hero.to(device)
        y.to(device)

        # print(x_ab)

        optimizer.zero_grad()
        output = model(x_ab, x_hero)
        loss = torch.nn.functional.binary_cross_entropy_with_logits(output, y)
        loss.backward()
        optimizer.step()

        pred = output > 0.

        # print("1", len(y))
        # print("2", (pred==y.bool()).sum())
        acc = (pred==y.bool()).sum().tolist() / len(y)
        # print(acc)


        if batch_idx % 50 == 0:
            print('Train Epoch: {} [{}/{} ({:.0f}%)]\tLoss: {:.6f}\tAcc: {:.2f}'.format(
                epoch, batch_idx * len(y), n_train,
                100. * batch_idx / n_train, loss.item(), acc))


def test(model, device, test_loader):
    model.eval()
    test_loss = 0
    correct = 0
    with torch.no_grad():

        for batch_idx, (x_ab, x_hero, y) in enumerate(test_loader):

            x_ab.to(device)
            x_hero.to(device)
            y.to(device)
            output = model(x_ab, x_hero)
            loss = torch.nn.functional.binary_cross_entropy_with_logits(output, y)

            pred = output > 0.
            acc = (pred==y.bool()).sum().tolist() / len(y)
            
            print('Test Acc: {:.2f}'.format(acc))

In [67]:
for e in range(10):
    train(model, device, train_loader, optimizer, e)
    test(model, device, test_loader)

Train Epoch: 0 [0/10000 (0%)]	Loss: 0.655009	Acc: 0.64
Train Epoch: 0 [5000/10000 (0%)]	Loss: 0.676205	Acc: 0.58
Test Acc: 0.58
Train Epoch: 1 [0/10000 (0%)]	Loss: 0.703956	Acc: 0.57
Train Epoch: 1 [5000/10000 (0%)]	Loss: 0.689299	Acc: 0.54
Test Acc: 0.60
Train Epoch: 2 [0/10000 (0%)]	Loss: 0.658305	Acc: 0.61
Train Epoch: 2 [5000/10000 (0%)]	Loss: 0.699775	Acc: 0.51
Test Acc: 0.59
Train Epoch: 3 [0/10000 (0%)]	Loss: 0.627236	Acc: 0.63
Train Epoch: 3 [5000/10000 (0%)]	Loss: 0.660549	Acc: 0.60
Test Acc: 0.60
Train Epoch: 4 [0/10000 (0%)]	Loss: 0.660170	Acc: 0.62
Train Epoch: 4 [5000/10000 (0%)]	Loss: 0.670358	Acc: 0.57
Test Acc: 0.60
Train Epoch: 5 [0/10000 (0%)]	Loss: 0.696905	Acc: 0.54
Train Epoch: 5 [5000/10000 (0%)]	Loss: 0.700769	Acc: 0.55
Test Acc: 0.61
Train Epoch: 6 [0/10000 (0%)]	Loss: 0.695045	Acc: 0.54
Train Epoch: 6 [5000/10000 (0%)]	Loss: 0.646289	Acc: 0.64
Test Acc: 0.62
Train Epoch: 7 [0/10000 (0%)]	Loss: 0.668298	Acc: 0.59
Train Epoch: 7 [5000/10000 (0%)]	Loss: 0.640892	A

In [64]:
class test_data(torch.utils.data.Dataset):
    def __init__(self):
        super(test_data, self).__init__()
        self.X_ab = list(range(10))
        self.X_hero = [(i,"i") for i in range(10)]
        self.Y = [[i,np.array([i,i])] for i in range(10)]
    
    def __len__(self):
        return 10
    
    def __getitem__(self, i):
        return self.X_ab[i], self.X_hero[i], self.Y[i]

In [66]:
tmp = torch.utils.data.DataLoader(test_data(), batch_size=2)
for a in tmp:
    print(a)
    break

[tensor([0, 1]), [tensor([0, 1]), ('i', 'i')], [tensor([0, 1]), tensor([[0, 0],
        [1, 1]], dtype=torch.int32)]]


In [62]:
a

[tensor([0, 1]),
 [tensor([0, 1]), ('i', 'i')],
 [tensor([0, 1]), [tensor([0, 1]), tensor([0, 1])]]]